# EuroSAT Matched-Protocol Comparison

This notebook reads the artifacts produced by `scripts/evaluate_eurosat_matched_protocol.py`.

Goal: compare the existing EuroSAT baselines and the 13-band pipeline under the same `80/10/10` split protocol without modifying the old baseline code or result files.

## Run The Evaluation Script First

From the project root:

```bash
python3 scripts/evaluate_eurosat_matched_protocol.py \
  --eurosat-root data/EuroSAT_MS \
  --clip-checkpoint checkpoints/ViT-B-16.pt
```

The script writes all matched-protocol outputs under `results/matched_protocol/`.

In [3]:
from pathlib import Path
import csv
import json

try:
    import pandas as pd
except ImportError:
    pd = None

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RESULTS_DIR = PROJECT_ROOT / 'results' / 'matched_protocol'

SUMMARY_CSV = RESULTS_DIR / 'eurosat_matched_protocol_summary.csv'
COMPARISON_CSV = RESULTS_DIR / 'eurosat_matched_protocol_comparison.csv'
SPLIT_CSV = RESULTS_DIR / 'eurosat_matched_protocol_split_manifest.csv'
MANIFEST_JSON = RESULTS_DIR / 'eurosat_matched_protocol_manifest.json'

for path in (SUMMARY_CSV, COMPARISON_CSV, SPLIT_CSV, MANIFEST_JSON):
    print(f'{path.name}:', 'FOUND' if path.exists() else 'MISSING')

eurosat_matched_protocol_summary.csv: FOUND
eurosat_matched_protocol_comparison.csv: FOUND
eurosat_matched_protocol_split_manifest.csv: FOUND
eurosat_matched_protocol_manifest.json: FOUND


In [4]:
def load_csv_rows(path: Path):
    with path.open(newline='') as f:
        return list(csv.DictReader(f))

summary_rows = load_csv_rows(SUMMARY_CSV)
comparison_rows = load_csv_rows(COMPARISON_CSV)
split_rows = load_csv_rows(SPLIT_CSV)
manifest = json.loads(MANIFEST_JSON.read_text())

print('Loaded rows:')
print('  summary    :', len(summary_rows))
print('  comparison :', len(comparison_rows))
print('  split rows :', len(split_rows))

Loaded rows:
  summary    : 1
  comparison : 4
  split rows : 27000


In [5]:
if pd is not None:
    summary_df = pd.DataFrame(summary_rows)
    comparison_df = pd.DataFrame(comparison_rows)
    display(summary_df)
    display(comparison_df)
else:
    print('Summary rows:')
    for row in summary_rows:
        print(row)
    print('\nComparison rows:')
    for row in comparison_rows:
        print(row)

,dataset,task,method,protocol,query_split,gallery_split,seed,num_train,num_queries,num_gallery,...,grad_clip,early_stop_tol,avg_query_fusion_ms,avg_query_optimization_ms,avg_gallery_fusion_ms,avg_gallery_optimization_ms,R@1,R@5,R@10,mAP
0,EuroSAT_MS,image_to_image_single_label,Ours (13-band pipeline),EuroSAT 80/10/10 matched protocol,val,test,42,21600,2700,2700,...,1.0,1e-06,2.7646088123502626,2.493848904193362,2.6358090213142096,2.39173719467578,83.55555534362793,96.03703618049622,98.00000190734863,45.55759329928292


,method,dataset,protocol,num_queries,num_gallery,R@1,R@5,R@10,mAP,source_artifact,notes,queries_vs_ours,gallery_vs_ours
0,PCA,EuroSAT_MS,EuroSAT 80/10/10 matched protocol,2700,2700,82.11110830307007,95.88888883590698,98.00000190734863,42.46266279783514,results/pca_baseline/pca_baseline_results.csv,existing baseline artifact,1.0,1.0
1,NDVI,EuroSAT_MS,EuroSAT 80/10/10 matched protocol,2700,2700,81.77777528762817,96.2592601776123,98.44444394111633,45.800061211127925,results/ndvi_baseline/ndvi_baseline_results.csv,existing baseline artifact,1.0,1.0
2,RS-TransCLIP,EuroSAT_MS,EuroSAT 80/10/10 matched protocol,2700,2700,76.85185074806213,91.96296334266663,95.333331823349,43.811700839687276,results/rs_transclip/rs_transclip_baseline_res...,existing baseline artifact; best_alpha=0.3,1.0,1.0
3,Ours (13-band pipeline),EuroSAT_MS,EuroSAT 80/10/10 matched protocol,2700,2700,83.55555534362793,96.03703618049622,98.00000190734863,45.55759329928292,results/matched_protocol/eurosat_matched_proto...,new matched-protocol 13-band pipeline evaluation,1.0,1.0


In [6]:
split_counts = {}
for row in split_rows:
    split_counts[row['split']] = split_counts.get(row['split'], 0) + 1

print('Split counts:', split_counts)
print('Protocol summary:')
print(json.dumps(manifest['protocol'], indent=2))

Split counts: {'train': 21600, 'val': 2700, 'test': 2700}
Protocol summary:
{
  "dataset": "EuroSAT_MS",
  "train_ratio": 0.8,
  "val_ratio": 0.1,
  "test_ratio": 0.1,
  "query_split": "val",
  "gallery_split": "test",
  "seed": 42
}
